In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
stocks = [
    "AAPL",  # Apple
    "GOOGL", # Google
    "META",  # Meta
    "AMZN",  # Amazon
    "MSFT",  # Microsoft
    "NVDA",  # Nvidia
    "JPM",   # JP Morgan
    "GS",    # Goldman Sachs
    "TSLA",  # Tesla
    "RIVN",  # Rivian
]

In [0]:
all_company_info = []

for ticker in stocks:
    try:
        stock = yf.Ticker(ticker)

        info = stock.info

        company_data = {
            'ticker'             : ticker,
            'company_name'       : info.get('shortName',None),
            'sector'             : info.get('sector',None),
            'industry'           : info.get('industry',None),
            'country'            : info.get('country', None),
            'full_time_employees': info.get('fullTimeEmployees', None),
            'market_cap'         : info.get('marketCap', None),
            'currency'           : info.get('currency', None),
            'exchange'           : info.get('exchange', None),
            'website'            : info.get('website', None),
            'ingestion_timestap' : datetime.today()
        }

        all_company_info.append(company_data)

    except Exception as e:
        print(f"Error fetching data for {ticker}: {e}")

company_df = pd.DataFrame(all_company_info)
display(company_df)
# print(company_df.columns.tolist())

In [0]:
company_spark_df = spark.createDataFrame(company_df)
company_spark_df.printSchema()

In [0]:
company_spark_df.write.format('delta').mode('append').option('mergeSchema','true').saveAsTable("raj.bronze.company_info")

In [0]:
verify_df = spark.table("raj.bronze.company_info")
display(verify_df)